# Higiene de demanda

La notebook 02 y el CLI usan `prepare_daily_demand`. Aquí se ve **qué cambia** respecto a agregar solo los días con ticket:

1. Calendario continuo: días sin venta → 0
2. Cap de cantidad diaria al percentil 99 (días con venta > 0)
3. Fuera el one-shot `23843` (PAPER CRAFT LITTLE BIRDIE, ~80k ud en un día)

Es una notebook de diagnóstico: no escribe artefactos.

> Requiere el paquete instalado en modo editable: `pip install -e .` desde la raíz del repo.


In [1]:
import pandas as pd

from inventario_ecommerce import config
from inventario_ecommerce.dataset import load_transactions
from inventario_ecommerce.features import (
    build_daily_sku_demand,
    clean_transactions,
    prepare_daily_demand,
)
from inventario_ecommerce.modeling.train import temporal_backtest_baseline

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.2f}".format)


## 1. Solo días con venta vs calendario

Sin rellenar ceros, rolling y media móvil solo ven días con transacción y sobreestiman la demanda diaria.


In [2]:
raw = load_transactions()
clean = clean_transactions(raw)

sparse = build_daily_sku_demand(clean)
dense = prepare_daily_demand(clean)

print(f"Filas solo-días-con-venta: {len(sparse):,}")
print(f"Filas calendario:          {len(dense):,}")
print(f"SKUs sparse: {sparse[config.COL_STOCK_CODE].nunique():,}")
print(f"SKUs dense:  {dense[config.COL_STOCK_CODE].nunique():,}")
print(f"Outliers excluidos: {sorted(config.OUTLIER_STOCK_CODES)}")

positive = dense.loc[dense["QuantitySold"] > 0, "QuantitySold"]
print(f"Máximo diario tras winsor P{config.DAILY_QTY_WINSOR_PERCENTILE:.0%}: {positive.max():,.1f}")


Filas solo-días-con-venta: 533,450
Filas calendario:          3,322,199
SKUs sparse: 4,907
SKUs dense:  4,906
Outliers excluidos: ['23843']
Máximo diario tras winsor P99%: 225.0


## 2. El mismo SKU en las dos versiones

`85123A` (WHITE HANGING HEART T-LIGHT HOLDER) es un fast mover: aun así, la media cae al contar los días sin venta.


In [3]:
sku = "85123A"
stock = config.COL_STOCK_CODE
cols = ["Date", stock, "QuantitySold"]

sparse_sku = sparse.loc[sparse[stock].astype(str) == sku, cols]
dense_sku = dense.loc[dense[stock].astype(str) == sku, cols]

print(f"Solo días con venta: {len(sparse_sku):,} filas | media {sparse_sku['QuantitySold'].mean():.2f} ud/día")
print(f"Calendario completo: {len(dense_sku):,} filas | media {dense_sku['QuantitySold'].mean():.2f} ud/día")
dense_sku.head(10)


Solo días con venta: 605 filas | media 165.54 ud/día
Calendario completo: 925 filas | media 77.08 ud/día


,Date,StockCode,QuantitySold
1216,2009-12-01,85123A,225.00
2973,2009-12-02,85123A,225.00
5008,2009-12-03,85123A,225.00
7232,2009-12-04,85123A,179.00
9483,2009-12-05,85123A,96.00
11810,2009-12-06,85123A,206.00
14263,2009-12-07,85123A,225.00
16808,2009-12-08,85123A,225.00
19428,2009-12-09,85123A,225.00
22119,2009-12-10,85123A,225.00


## 3. Efecto en el backtest

Mismo holdout de 30 días. El MAE "sparse" solo puntúa días con ticket, así que compara predicción contra la demanda de los días buenos.


In [4]:
_, sparse_global = temporal_backtest_baseline(sparse, horizon_days=30, lookback_days=30)
_, dense_global = temporal_backtest_baseline(dense, horizon_days=30, lookback_days=30)

comparison = pd.concat(
    [
        sparse_global.assign(series="solo_dias_con_venta"),
        dense_global.assign(series="calendario_con_ceros"),
    ],
    ignore_index=True,
)
comparison[["series", "GlobalMAE", "GlobalMAPE", "CutoffDate", "EvalStartDate", "EvalEndDate"]]


,series,GlobalMAE,GlobalMAPE,CutoffDate,EvalStartDate,EvalEndDate
0,solo_dias_con_venta,20.19,283.75,2011-11-09,2011-11-10,2011-12-09
1,calendario_con_ceros,4.06,142.74,2011-11-09,2011-11-10,2011-12-09


Los dos MAE no son comparables entre sí: miden sobre poblaciones de días distintas. Para reponer inventario hay que decidir también los días en los que no se vende, así que el pipeline usa el calendario completo.

MAPE queda de referencia, no como métrica de decisión: con muchos días a cero en el denominador se dispara.
